# Go1 IMU sensitivity — Colab runnerRuns the two GPU-bound steps of the IMU edge-case project:1. **Train** a Go1 joystick policy (stock DR, and a no-noise variant).2. **OFAT sweep** — inject idealised IMU errors one at a time and find where   performance knees, then compare each knee against a real BMI270.The sweep is the decision point. If no channel's knee is reachable by a realdevice, the project's premise is false for this task and we learn it hererather than in week 3.**Set Runtime → Change runtime type → GPU before running anything.**

## 1. GPU checkFail loudly rather than silently training on CPU for hours.

In [ ]:
import subprocessprint(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

## 2. InstallThe CUDA JAX wheel is the part people get wrong — the CPU wheel installs fine and runs 100x slower with no warning.

In [ ]:
!pip install -q -U "jax[cuda12]"!pip install -q mujoco mujoco-mjx brax playground

In [ ]:
import jaxdevs = jax.devices()print("devices:", devs)assert any(d.platform == "gpu" for d in devs), (    "No GPU visible to JAX. Runtime > Change runtime type > GPU, then "    "Runtime > Restart session, then re-run this cell.")print("OK — JAX sees a GPU")

## 3. Mount Drive and cloneCheckpoints go to Drive. A Colab disconnect wipes the VM; a three-hourtraining run goes with it unless the weights are somewhere persistent.Replace `REPO_URL` with your GitHub remote. For a private repo use`https://<TOKEN>@github.com/<user>/imu-diff-sim.git`.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import pathlibOUT = pathlib.Path('/content/drive/MyDrive/imu-diff-sim-runs')OUT.mkdir(parents=True, exist_ok=True)print("checkpoints ->", OUT)

In [ ]:
REPO_URL = "https://github.com/<your-user>/imu-diff-sim.git"   # <-- edit!git clone -q $REPO_URL /content/imu-diff-sim || echo "already cloned"%cd /content/imu-diff-sim!git pull -q!pip install -q -e .

## 4. Sanity: the model still passes its tests on GPUCheap, and catches a bad install before an hour of training.

In [ ]:
!python -m pytest -q --no-header -x

## 5. TrainTwo policies, because the comparison needs both:- `stock` — the default `noise_config.level = 1.0`, i.e. uniform gyro DR of  ±0.2 rad/s. This is standard practice, and it is 146× a real BMI270.- `clean` — `level = 0.0`. Needed so the OFAT sweep measures injected errors  against a clean floor rather than on top of noise 146× larger.Start with a reduced `--timesteps` to check throughput before committing to afull run. Watch the reward: Go1 joystick should climb steadily; if it is flatafter a few million steps something is wrong with the install.

In [ ]:
# Throughput check first — kill it once you can estimate steps/sec.!python scripts/train_go1.py --timesteps 2000000 --noise-level 0.0 \    --out /content/drive/MyDrive/imu-diff-sim-runs/go1_clean_smoke.pkl

In [ ]:
# Full runs. Expect tens of minutes each on a decent GPU.!python scripts/train_go1.py --noise-level 0.0 \    --out /content/drive/MyDrive/imu-diff-sim-runs/go1_clean.pkl

In [ ]:
!python scripts/train_go1.py --noise-level 1.0 \    --out /content/drive/MyDrive/imu-diff-sim-runs/go1_stock.pkl

## 6. OFAT sweep — the actual questionFive factors, swept one at a time. The two that matter most are `gyro_white`and `gyro_bias` at **equal magnitude**: white noise the policy can averageaway over an episode (as 1/√N), a persistent bias it cannot. If bias degradesperformance at far lower magnitude, the structural argument is confirmedempirically instead of asserted.

In [ ]:
!python scripts/ofat_sensitivity.py \    --policy /content/drive/MyDrive/imu-diff-sim-runs/go1_clean.pkl \    --episodes 64 --horizon 500 --points 9 \    --out /content/drive/MyDrive/imu-diff-sim-runs/ofat_clean.npz

In [ ]:
# Same sweep against the stock-DR policy. Expect it to tolerate far more —# that difference is the cost of the robustness margin.!python scripts/ofat_sensitivity.py \    --policy /content/drive/MyDrive/imu-diff-sim-runs/go1_stock.pkl \    --episodes 64 --horizon 500 --points 9 \    --out /content/drive/MyDrive/imu-diff-sim-runs/ofat_stock.npz

## 7. Read the resultThe decision is the gap between each knee and the BMI270 line.

In [ ]:
import numpy as np, matplotlib.pyplot as pltfrom imu_diff_sim.sim.go1_obs import (    BMI270_GYRO_WHITE_STD_50HZ, PLAYGROUND_GYRO_DR_STD)d = np.load('/content/drive/MyDrive/imu-diff-sim-runs/ofat_clean.npz')factors = sorted({k.split('__')[0] for k in d.files})fig, axes = plt.subplots(1, len(factors), figsize=(4*len(factors), 4),                         constrained_layout=True)for ax, f in zip(np.atleast_1d(axes), factors):    m, r = d[f'{f}__magnitudes'], d[f'{f}__return']    ax.plot(m, r / max(r.max(), 1e-9), 'o-')    if f.startswith('gyro') and f != 'gyro_scale':        ax.axvline(BMI270_GYRO_WHITE_STD_50HZ, color='green', ls='--',                   label='BMI270')        ax.axvline(PLAYGROUND_GYRO_DR_STD, color='crimson', ls=':',                   label='Playground DR')        ax.legend(fontsize=7)    if f != 'latency':        ax.set_xscale('log')    ax.set_title(f); ax.set_xlabel('magnitude')    ax.set_ylabel('normalised return'); ax.grid(alpha=.3)plt.show()